In [2]:
import os
# os.environ["HF_HOME"] = "/home/seungwoochoi/data/huggingface/cache"
from tqdm import tqdm
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from FlagEmbedding import BGEM3FlagModel
import numpy as np
device = "auto"
np.set_printoptions(threshold=np.inf)
import matplotlib.pyplot as plt

/opt/miniconda3/envs/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embedding_model = BGEM3FlagModel('BAAI/bge-m3', devices=device)

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]Error while downloading from https://cdn-lfs-us-1.hf.co/repos/23/2c/232ca60237b0bb19bb6c28c5a6c8af79f2e423333a9626aad445543b80fbf31e/b5e0ce3470abf5ef3831aa1bd5553b486803e83251590ab7ff35a117cf6aad38?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27pytorch_model.bin%3B+filename%3D%22pytorch_model.bin%22%3B&response-content-type=application%2Foctet-stream&Expires=1756435185&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1NjQzNTE4NX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzIzLzJjLzIzMmNhNjAyMzdiMGJiMTliYjZjMjhjNWE2YzhhZjc5ZjJlNDIzMzMzYTk2MjZhYWQ0NDU1NDNiODBmYmYzMWUvYjVlMGNlMzQ3MGFiZjVlZjM4MzFhYTFiZDU1NTNiNDg2ODAzZTgzMjUxNTkwYWI3ZmYzNWExMTdjZjZhYWQzOD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eXBlPSoifV19&Signature=ps2KtuoXYl7Piv8gS67ras2A5pxLLpL%7E1ObO%7Ev-172WQTaI39bZFnwofDkzt7xdrojOI53JKXM2bZoK%7EElGCOv3XfXju9MBweGLMkE-13sviIipl0Mmx

ReadTimeout: (ReadTimeoutError("HTTPSConnectionPool(host='cdn-lfs-us-1.hf.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 180712b4-8506-445e-82df-aa8b5d515378)')

In [4]:
import pandas as pd 
ambiguous_words_df = pd.read_csv("data/ambiguous_words_paired.csv")

In [5]:
print(list(ambiguous_words_df['Meaning 1']))
# embedding_model.encode(list(ambiguous_words_df['Meaning 1']))
meaning1_embedding = embedding_model.encode(list(ambiguous_words_df['Meaning 1']))['dense_vecs']
meaning2_embedding = embedding_model.encode(list(ambiguous_words_df['Meaning 2']))['dense_vecs']
def1_embedding = embedding_model.encode(list(ambiguous_words_df['Def1']))['dense_vecs']


['Bank, a financial institution', 'Bat, a flying mammal', 'Bark, the sound a dog makes', 'Spring, a season', 'Jam, a fruit preserve', 'Right, correct', 'Light, illumination', 'Match, a competition', 'Watch, to observe', 'Rock, a stone', 'Ring, a piece of jewelry', 'Seal, a marine animal', 'Current, the flow of water', 'File, a folder for documents', 'Nail, a fastener', 'Can, a container', 'Well, a water source', 'Point, a sharp tip', 'Trip, a journey', 'Row, a line of items']


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [6]:
print((meaning1_embedding @ meaning2_embedding.T).diagonal())
print((meaning1_embedding @ def1_embedding.T).diagonal())

print(np.mean((meaning1_embedding @ meaning2_embedding.T).diagonal()))
print(np.mean((meaning1_embedding @ def1_embedding.T).diagonal()))


[0.746  0.7744 0.7085 0.8286 0.6074 0.822  0.697  0.7236 0.6846 0.7495
 0.7964 0.64   0.712  0.692  0.7715 0.758  0.696  0.825  0.756  0.7466]
[0.6763 0.689  0.7974 0.6943 0.568  0.594  0.6655 0.792  0.762  0.516
 0.7754 0.6377 0.7266 0.753  0.608  0.6543 0.577  0.7495 0.8335 0.7476]
0.737
0.691


# Synonyms